# 06 · Which number to put in a report

**What this teaches:** which summary column answers which question, how the columns relate, and how each is misread. Each column is defined in the [data dictionary](../docs/api/data_dictionary.md); what is not written there is how they move against each other, which is what you need before you put one of them in a report.

| The question | The column |
|---|---|
| How likely are any deaths? | `sb_p_gt0` |
| How likely is a serious month? | `sb_p_gt25`, `sb_p_gt100`, `sb_p_gt500`, `sb_p_gt1000` |
| How many deaths to expect, all things considered? | `sb_mean` |
| How many deaths if there are any? | `sb_mean_if_any`, `sb_median_if_any` |
| What could a bad month look like? | `sb_severe_scenario` |
| How many draws are the numbers from? | `n_draws` |

**Why one number is not enough.** In most places and months the most likely outcome is no deaths at all, but some draws have many. A single number either hides the chance (an expected 0.3 deaths can be a 3% chance of 10) or hides the size. The usual pair for a report is **how likely** and **how bad if it happens**.

Every relation below is measured on the data served, not asserted. **You need an API key** (set in `.env`, see `notebooks/README.md`).

## 1 · Setup

In [ ]:
import math
import os
import tempfile
import time

import matplotlib.pyplot as plt
import pandas as pd
import requests
from dotenv import load_dotenv

from views_crafdapi.client import CrafdApiClient
from views_crafdapi.forecast.serialize.schema import SEVERE_MIN_DRAWS, SEVERE_TAIL   # the service's own rule
from views_crafdapi.plotting import plot_pixel_grid

load_dotenv()  # reads CRAFD_KEY (your key) and, optionally, VIEWS_API_URL

BASE_URL = os.environ.get("VIEWS_API_URL", "https://crafdapi.viewsforecasting.org")
API_KEY = os.environ.get("CRAFD_KEY", "").strip().strip('"')
HEADERS = {"X-API-Key": API_KEY}  # for the bulk download, which is a plain file request

client = CrafdApiClient(BASE_URL, API_KEY, timeout=300)
print("Base URL:", BASE_URL)
print("API key :", "set" if API_KEY else "MISSING: set it in .env")


def download_bulk(level, outcome, months=None, columns=None):
    """One bulk file as a DataFrame, and the month the forecast was made in. The file is saved to
    a temporary file first; with `months` and `columns`, only those are read from it. A busy
    server can answer 502 or 504, the service answers 503 for a minute or so while it loads a new
    forecast, and a slow connection can time out or drop: two retries, a minute apart."""
    for attempt in (1, 2, 3):
        try:
            resp = requests.get(f"{BASE_URL}/bulk/{level}/{outcome}", headers=HEADERS, timeout=600, stream=True)
            if resp.status_code in (502, 503, 504) and attempt < 3:
                resp.close()
                print(f"The server answered {resp.status_code}; trying again in 60 seconds...")
                time.sleep(60)
                continue
            if not resp.ok:
                detail = resp.text[:300]
                resp.close()
                raise SystemExit(f"/bulk/{level}/{outcome} answered {resp.status_code}: {detail}")
            out = tempfile.NamedTemporaryFile(suffix=".parquet", delete=False)
            try:
                with resp, out:
                    for chunk in resp.iter_content(chunk_size=1 << 20):
                        out.write(chunk)
            except BaseException:
                os.unlink(out.name)   # never leave a partial file behind
                raise
            break
        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError,
                requests.exceptions.ChunkedEncodingError):
            if attempt == 3:
                raise
            print("The download was cut off; trying again in 60 seconds...")
            time.sleep(60)
    made_in = resp.headers.get("X-Forecast-Made-In")
    try:
        if not made_in:
            raise SystemExit(f"/bulk/{level}/{outcome} did not say which month the forecast was made in.")
        filters = [("month", "in", list(months))] if months else None
        table = pd.read_parquet(out.name, columns=columns, filters=filters)
    finally:
        os.unlink(out.name)
    return table, made_in

In [ ]:
if not API_KEY or API_KEY == "your-api-key-here":
    raise SystemExit("No usable API key. Set CRAFD_KEY in .env (see notebooks/README.md), restart "
                     "the kernel and run again.")
try:
    health = client.health()
except Exception as e:   # an error answer, a timeout or a dropped connection
    health = {}
    print("Health check unavailable, continuing:", e)
RUN = health.get("served_run") or "not reported"

# The first forecast month, at grid level: every summary column, for every cell.
countries, MONTH = download_bulk("country", "sb")
cells, cells_month = download_bulk("grid", "sb", [MONTH])
if cells_month != MONTH:   # a new forecast arrived between the downloads
    raise SystemExit("The two files come from different forecasts. Run this cell again.")
nga_rows = countries.loc[countries["country_iso3"] == "NGA", "country_code"]
NIGERIA = int(nga_rows.iloc[0]) if len(nga_rows) else None   # the example country only
CHANCES = [c for c in cells.columns if c.startswith("sb_p_gt")]   # read from the data, not typed
print(f"Run {RUN}, month {MONTH}: {len(cells):,} grid cells; chance columns: {', '.join(CHANCES)}")

## 2 · The columns side by side

Nigeria's ten cells with the highest chance of any deaths. Read across a row: a chance, then sizes.

In [ ]:
SIDE_BY_SIDE = ["priogrid_id", "admin1_name", "sb_p_gt0", "sb_p_gt25", "sb_mean", "sb_mean_if_any",
                "sb_median_if_any", "sb_severe_scenario", "n_draws"]
example = cells[cells["country_code"] == NIGERIA] if NIGERIA is not None else cells
example.nlargest(10, "sb_p_gt0")[SIDE_BY_SIDE]

## 3 · Expected deaths are the chance times the size

`sb_mean` averages over all draws; `sb_mean_if_any` over the draws above 0 (with any deaths); `sb_p_gt0` is the share of draws above 0. So, when no draw is negative,

> `sb_mean` = `sb_p_gt0` × `sb_mean_if_any`

and the one number for "how many to expect" splits into "how likely" and "how bad if so". Where no draw has a death, `sb_mean_if_any` is empty (there is nothing to average) and `sb_mean` is 0. The cell below counts the cells where the identity does not hold, rather than assuming there are none.

In [ ]:
has_draws = cells["n_draws"] > 0
product = cells["sb_p_gt0"] * cells["sb_mean_if_any"].fillna(0)   # 0 where no draw had a death
gap = (product - cells["sb_mean"]).abs()
breaks = has_draws & (gap > 1e-5 * cells["sb_mean"].abs().clip(lower=1))   # float32 rounding
print(f"Cells with draws: {int(has_draws.sum()):,}; with any chance of deaths: {int((cells['sb_p_gt0'] > 0).sum()):,}")
print(f"Cells where sb_mean is not sb_p_gt0 x sb_mean_if_any: {int(breaks.sum())}")
print(f"Cells with no death in any draw but an expected number above 0: "
      f"{int((has_draws & (cells['sb_p_gt0'] == 0) & (cells['sb_mean'] > 0)).sum())}")

## 4 · Likely but small, unlikely but large

The pair disagrees, and the disagreement is the point. Split the cells with any chance of deaths in four, at the medians of the two columns:

- **likely but small**: something will probably happen, and it will probably be minor. Easy to over-react to if you read the chance alone.
- **unlikely but large**: probably nothing, but if it happens it is severe. **A map of the chance hides this group and a map of the size exaggerates it**, and it is usually the group a contingency plan is for.

The medians describe this forecast, not a convention. A cell at a median counts as at or above it, so many cells with the same chance fall on one side.

In [ ]:
at_risk = cells[cells["sb_p_gt0"] > 0].dropna(subset=["sb_mean_if_any"]).copy()
p_cut, size_cut = at_risk["sb_p_gt0"].median(), at_risk["sb_mean_if_any"].median()
likely, large = at_risk["sb_p_gt0"] >= p_cut, at_risk["sb_mean_if_any"] >= size_cut
at_risk["group"] = "unlikely and small"
at_risk.loc[likely & large, "group"] = "likely and large"
at_risk.loc[likely & ~large, "group"] = "likely but small"
at_risk.loc[~likely & large, "group"] = "unlikely but large"
print(f"{len(at_risk):,} cells with any chance of deaths; medians: chance {p_cut:.3f}, deaths if any {size_cut:.1f}")
print(at_risk["group"].value_counts().reindex(
    ["likely and large", "likely but small", "unlikely but large", "unlikely and small"], fill_value=0).to_string())

## 5 · The average or the middle of the deaths if any?

`sb_mean_if_any` and `sb_median_if_any` describe the same draws: those above 0. When the deaths, if any, have a long tail (most draws few, some many), the average sits above the middle value. With few such draws the two are often equal. The cell below counts which way they differ in this forecast, rather than assuming.

Which to report is a choice about errors: the average is the best single number if large misses cost more than small ones in proportion to their square, the median if every death of error costs the same. For a risk report the average is the usual choice, because the median reports the smaller number exactly where the tail is long. Say which one you used.

In [ ]:
both = cells.dropna(subset=["sb_mean_if_any", "sb_median_if_any"])
above = int((both["sb_mean_if_any"] > both["sb_median_if_any"]).sum())
equal = int((both["sb_mean_if_any"] == both["sb_median_if_any"]).sum())
below = int((both["sb_mean_if_any"] < both["sb_median_if_any"]).sum())
print(f"Cells with both: {len(both):,}; the average is above the median in {above:,}, equal in {equal:,}, below in {below:,}")

## 6 · What the severe case averages

`sb_severe_scenario` is the average of the worst 5% of the draws: of the worst `ceil(0.05 × n_draws)` draws. So what it averages depends on how many draws a row has. Below 20 usable draws the column is empty, because the worst 5% would be a single draw. At exactly 20 draws it is still a single draw (the service fills it; being corrected in views-crafdapi #286); from 21 draws on it averages at least two.

In [ ]:
draw_counts = cells.loc[cells["n_draws"] > 0, "n_draws"].astype(int).value_counts().sort_index()
for n, rows in draw_counts.items():
    worst = math.ceil(SEVERE_TAIL * n)
    if n < SEVERE_MIN_DRAWS:
        print(f"{rows:,} cells with {n:,} draws: the severe case is empty (the worst 5% would be {worst} draw)")
    else:
        print(f"{rows:,} cells with {n:,} draws: the severe case averages the worst {worst} of them"
              + (" (a single draw)" if worst == 1 else ""))
print(f"Cells with an empty severe case: {int(cells['sb_severe_scenario'].isna().sum()):,}")

## 7 · The thresholds

The chances of more than 0, 25, 100, 500 and 1,000 deaths never rise as the number rises; the cell below counts any row where one does. 25 deaths in a month is the level at which VIEWS counts a country as in conflict, and one of the edges of the bins in its forecast scoring; it is a good default for "a serious month". The data dictionary says where the others come from.

A chance of 0 means no draw went above that number. It does not mean impossible: the smallest chance above 0 a row can show is one over its `n_draws`.

In [ ]:
for column in CHANCES:
    print(f"{column:12}: above 0 in {int((cells[column] > 0).sum()):>7,} cells, "
          f"0.5 or more in {int((cells[column] >= 0.5).sum()):>6,}")
by_threshold = sorted(CHANCES, key=lambda c: int(c.rsplit("gt", 1)[1]))   # 0, 25, 100, ... not as text
rises = (cells[by_threshold].diff(axis=1) > 1e-12).any(axis=1)
print(f"Rows where a chance rises as the threshold rises: {int(rises.sum())}")

## 8 · Three maps, three scales

| Question | Column | Scale |
|---|---|---|
| Where could deaths happen? | `sb_p_gt0` | linear: a chance is already between 0 and 1 |
| How bad there, if they do? | `sb_mean_if_any` | log: deaths span several orders of magnitude |
| How many to expect? | `sb_mean` | log |

The first and third maps show every cell with draws, 0 included. The second leaves empty the cells where no draw was above 0: a 0 there would claim "no deaths if violence occurs". The three maps are not on one colour scale, because one is a chance and two are counts.

In [ ]:
draw = cells.dropna(subset=["latitude", "longitude"])
for column, transform, title in (
        ("sb_p_gt0", "none", "Where: the chance of any state-based deaths"),
        ("sb_mean_if_any", "log1p", "How bad there: state-based deaths if there are any (log scale)"),
        ("sb_mean", "log1p", "How many to expect: expected state-based deaths (log scale)")):
    shown = draw.dropna(subset=[column])
    fig, ax, _ = plot_pixel_grid(shown, lon_col="longitude", lat_col="latitude", value_col=column,
                                 transform=transform, cmap="YlOrRd", figsize=(14, 7))
    ax.set_title(f"{title}, {MONTH}  (run {RUN})")
    plt.show()

## 9 · Common misreadings

| Misreading | Instead |
|---|---|
| "`sb_mean` is 0.3, so nothing will happen." | It can be a small chance of a large number. Read `sb_p_gt0` with `sb_mean_if_any`. |
| "`sb_mean_if_any` is 40, so expect 40 deaths." | Only if deaths happen at all; multiply by `sb_p_gt0` for the expected number (section 3). |
| "A chance of 0 means it cannot happen." | No draw went above that number; the smallest chance shown is one over `n_draws`. |
| "An empty value is 0." | Empty means no answer, not zero: the row has no usable draws; or, for `sb_mean_if_any` and `sb_median_if_any`, no draw above 0; or, for `sb_severe_scenario`, fewer than 20 draws. |
| "The country's chance is the average of its cells'." | It is the chance that the country's total passes the threshold; take it from the country file (`05_totals.ipynb`). |
| "The severe case is the worst that can happen." | It is the average of the worst 5% of draws: a bad month that is realistic, not the worst conceivable. |